# qdebug quickstart: logic bugs vs. noise on the IonQ simulator

Records a *golden* behaviour + noise baseline for a circuit, then classifies later
edits as **PASS / REFACTOR_OK / LOGIC_BUG / NOISE_REGRESSION**.

Run top to bottom. With no `IONQ_API_KEY` it uses the local simulator; with a key
it runs on IonQ's cloud simulator (ideal + hardware noise model).

This effort is supported by Qollab & IonQ.

In [ ]:
# 1) Install (skip what your environment already has)
%pip install -q qiskit qiskit-aer qiskit-ionq matplotlib

In [ ]:
# 2) Make the qdebug package importable.
#    Put this notebook inside the repo's notebooks/ folder, or upload the repo
#    next to it, or `%pip install -e /path/to/qdebug_project`.
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import qdebug
print("qdebug", qdebug.__version__)

In [ ]:
# 3) Pick a backend. Default is local Aer so Run-all works with no prompt
#    (Qollab judging uses a free local simulator). IonQ is optional: set
#    IONQ_API_KEY in the environment before this cell. Do not paste a key here.
BACKEND = "ionq" if os.environ.get("IONQ_API_KEY") else "local"
backend = qdebug.make_backend(BACKEND, noise_model="forte-1")
print(backend.describe())

In [ ]:
# 4) Smoke test: connectivity + bit ordering (expects '01' for X on qubit 0)
from qiskit import QuantumCircuit
from qdebug.circuits import probes
qc = QuantumCircuit(2, name="doctor"); qc.x(0)
pc = probes(qc, ("Z",))
print("ideal:", backend.ideal_probs(pc)[0])
print("noisy:", backend.noisy_counts(pc, 200, seed=1)[0])

In [ ]:
# 5) Record golden snapshots (1 ideal job + N seeded noisy jobs per circuit)
from qdebug import record
from qdebug.demo_suite import golden_circuits, mutants

snaps = {n: record(n, c, backend, shots=1000, n_seeds=3) for n, c in golden_circuits().items()}
for n, s in snaps.items():
    print(f"{n:8s} noise baseline {s.noise.mean:.4f} +/- {s.noise.std:.4f}")

In [ ]:
# 6) Check modified circuits: some harmless, some buggy, one just noisier
from qdebug import check
from qdebug.report import format_report, summary_table

reports, expected = [], {}
for m in mutants():
    r = check(m.name, m.circuit, snaps[m.base], backend)
    reports.append(r); expected[m.name] = m.expected
print(summary_table(reports, expected))

In [ ]:
# 7) Why did one fail? Full explanation + blamed edit
for r in reports:
    if r.name in ("ghz4_cx_flipped", "bell_tiny_phase_drift", "ghz4_cx_padding"):
        print(format_report(r)); print()

In [ ]:
# 8) Logic-vs-noise map
from qdebug.report import plot_reports
from IPython.display import Image, display
png = plot_reports(reports, "qdebug_map.png")
display(Image(png)) if png else print("matplotlib not installed")

## Use it on your own circuit

```python
snap = record("my_circuit", my_golden_circuit, backend)
report = check("my_circuit", my_edited_circuit, snap, backend)
print(format_report(report))
```
Circuits must be unitary (no mid-circuit measurement); measurements are added for you.